# Experiment: Berthoud Residual U-Net V0

Objective: train the first residual U-Net on the Berthoud v0 NPZ dataset and test whether ML-corrected mass-solver winds are closer to WindNinja momentum-solver winds than raw mass-solver winds.

Success criterion: test-set vector RMSE improves over the mass-solver baseline.

## Paths

This notebook expects the repo and processed dataset to already exist. Do not run WindNinja data generation from Colab for v0; copy the processed NPZ dataset into Drive from the laptop workflow.

In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = Path('/content').exists()
REPO_DIR = Path('/content/mountain_windninja') if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path('/content/drive/MyDrive/windninja_ml') if IN_COLAB else REPO_DIR / 'ml/residual_unet/outputs/colab_local'
DRIVE_DATA = DRIVE_ROOT / 'data/processed/berthoud_v0'
LOCAL_DATA = Path('/content/data/berthoud_v0') if IN_COLAB else REPO_DIR / 'ml/residual_unet/data/processed/berthoud_v0'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
LOG_CSV = DRIVE_ROOT / 'logs/train_log.csv'

REPO_DIR, DRIVE_ROOT, LOCAL_DATA

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_CSV.parent.mkdir(parents=True, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

## Install And Stage Data

Training should read from local Colab disk, not directly from many small files in Google Drive.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'ml/residual_unet/requirements.txt')],
    check=True,
)

if IN_COLAB:
    if LOCAL_DATA.exists():
        shutil.rmtree(LOCAL_DATA)
    if not DRIVE_DATA.exists():
        raise FileNotFoundError(f'Missing processed dataset in Drive: {DRIVE_DATA}')
    shutil.copytree(DRIVE_DATA, LOCAL_DATA)

assert (LOCAL_DATA / 'manifest.csv').exists(), LOCAL_DATA
assert (LOCAL_DATA / 'normalization.json').exists(), LOCAL_DATA
print(f'Training data staged at: {LOCAL_DATA}')

## Train

Checkpoints are written to Drive every epoch as `latest.pt` and `best.pt`.

In [ ]:
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR)
cmd = [
    sys.executable,
    '-m',
    'ml.residual_unet.train',
    '--config',
    str(REPO_DIR / 'ml/residual_unet/configs/berthoud_v0.yaml'),
    '--data',
    str(LOCAL_DATA),
    '--checkpoint-dir',
    str(CHECKPOINT_DIR),
    '--log-csv',
    str(LOG_CSV),
]
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)

## Evaluate

The evaluation writes metrics and a few residual/error-reduction plots to Drive.

In [ ]:
EVAL_OUT = DRIVE_ROOT / 'eval/berthoud_v0'
cmd = [
    sys.executable,
    '-m',
    'ml.residual_unet.evaluate',
    '--checkpoint',
    str(CHECKPOINT_DIR / 'best.pt'),
    '--data',
    str(LOCAL_DATA),
    '--out',
    str(EVAL_OUT),
]
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)

In [ ]:
import json

metrics = json.loads((EVAL_OUT / 'metrics.json').read_text())
metrics